# ✈️ Airbus — Maintenance Prédictive
### NASA CMAPSS | Sklearn + PyTorch LSTM + Plotly Express

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import kagglehub, os
print("✅ Librairies OK")

In [ ]:
path = kagglehub.dataset_download("behrad3d/nasa-cmaps")
cols = ["engine_id","cycle"]+[f"op_{i}" for i in range(1,4)]+[f"sensor_{i}" for i in range(1,22)]
df = pd.read_csv(os.path.join(path,"train_FD001.txt"), sep=" ", header=None, names=cols)
df.dropna(axis=1, inplace=True)
max_c = df.groupby("engine_id")["cycle"].max().reset_index()
max_c.columns = ["engine_id","max_cycle"]
df = df.merge(max_c, on="engine_id")
df["RUL"] = df["max_cycle"] - df["cycle"]
print(f"✅ {df.shape[0]:,} mesures | {df.engine_id.nunique()} moteurs")

In [ ]:
fig = px.line(df[df.engine_id.isin([1,2,3,4,5])], x="cycle", y="sensor_11", color="engine_id",
    title="🔧 Dégradation Capteur 11 — Importance 35.4%", template="plotly_dark")
fig.show()

In [ ]:
fig = px.histogram(df[df.cycle==1], x="max_cycle", nbins=30,
    title="⚠️ Distribution Durée de Vie des Moteurs", template="plotly_dark",
    color_discrete_sequence=["#FF6B6B"])
fig.add_vline(x=df.max_cycle.mean(), line_dash="dash", line_color="yellow", annotation_text=f"Moyenne: {df.max_cycle.mean():.0f} cycles")
fig.show()

In [ ]:
results = {"Modele":["LSTM","Gradient Boosting","LightGBM","Random Forest","XGBoost","KNN","Lineaire"],"RMSE":[17.40,18.90,18.97,19.07,19.91,20.52,21.69],"Type":["Deep Learning","Ensemble","Ensemble","Ensemble","Ensemble","Instance","Lineaire"]}
fig = px.bar(pd.DataFrame(results).sort_values("RMSE"), x="Modele", y="RMSE", color="Type",
    title="🏆 Comparaison Modeles ML — RMSE", template="plotly_dark", text="RMSE")
fig.update_traces(texttemplate="%{text:.2f}", textposition="outside")
fig.show()